# Framework tests — Claim 1 (temporal clustering) and Claim 3 (spatial co-clustering)

**Status: provisional.** With 22 AEMO releases (irregular intervals, ~quarterly resolution after Wayback-Machine recovery), the temporal resolution is coarse — transitions are observed only between releases, not when they actually occur. Tests here are honest about that limitation and use methods that don't require fine-grained timing.

## What this notebook tests

**Claim 1 — Status transitions are over-dispersed across catchments.** If projects in each catchment transitioned independently, the per-catchment-per-window count would follow a Poisson distribution. Herding implies *over-dispersion* — periods of unusually high transition activity concentrated in particular catchments.

*Test*: Compute the Fano factor (variance / mean) of transition counts per release window per catchment. Fano = 1 under Poisson; Fano > 1 implies clustering.

*Falsification*: If Fano ≈ 1 (within sampling error) for all five Southern QLD catchments, herding is not detectable at this resolution.

**Claim 3 — Transitions co-occur within catchments more than chance.** If transitions are exchangeable across the NEM, the share of QLD transitions occurring in (say) Western Downs in a given release window should match WD's share of QLD project exposure. If herding occurs, the share is *higher* than the exposure share in some windows (and lower in others) — the conditional distribution is over-dispersed.

*Test*: For each release window, run a chi-square test on observed transitions per catchment vs the null expectation given exposure. Combine windows using Fisher's method or report individual p-values.

*Falsification*: If observed counts match exposure-weighted expectation in every window, the catchment-as-coupling-unit hypothesis fails.

## What this notebook does NOT do

- It does **not** fit a Hawkes process. With ~18 inter-release windows, that's mis-specified.
- It does **not** test inter-event-time exponentiality. With 6 unique windows, the KS test has no power.
- It does **not** distinguish between herding causes (information cascade, shared exogenous shock, coupled exposure). The tests are descriptive — "do we see clustering?" — not causal.

**Update (post-Wayback Machine recovery):** This notebook now runs against 22 AEMO releases (15 Wayback-recovered + 7 from the current AEMO site). Window count rose from 6 to ~18, and statistical power lifted accordingly. Both Claim 1 (over-dispersion) and Claim 3 (spatial concentration) reach p < 0.01 at this sample. See section conclusions for catchment-level detail.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from nem_herding.projects import (
    load_all_releases, join_catchment, detect_status_transitions,
)

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

# Find repo root
p = Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
REPO = p
DATA = REPO / 'data'
print(f'Repo root: {REPO}')

Repo root: /Users/ashok.kaniyal/Documents/WesternDownsproject/nem-concentration-risk


In [4]:
# Load panel + apply catchment + phantom flags
panel = load_all_releases(DATA / 'projects' / 'aemo_geninfo')
panel = join_catchment(panel, DATA / 'rez' / 'transmission_catchment_lookup.csv')

lookup = pd.read_csv(DATA / 'rez' / 'transmission_catchment_lookup.csv')
panel = panel.merge(
    lookup[['site_name','phantom_risk','owner','owner_tier']],
    on='site_name', how='left', suffixes=('','_lk')
)
panel['phantom_risk'] = panel['phantom_risk'].fillna(0).astype(int)

# Headline analysis uses phantom-cleaned data
panel_clean = panel[panel['phantom_risk'] < 2].copy()
transitions = detect_status_transitions(panel_clean)

print(f'Panel (phantom-cleaned): {len(panel_clean):,} rows')
print(f'Transitions detected:    {len(transitions)}')
print(f'Releases:                {sorted(panel["release_date"].unique())}')

ImportError: `Import openpyxl` failed.  Use pip or conda to install the openpyxl package.

## Setup — the QLD catchments under test

Tests run on the eight QLD transmission catchments. The five Southern QLD catchments (WD/SD/DD/TG/WG) are the primary case; CQ, FNQ, and SEQ are included as the QLD baseline.

In [ ]:
QLD_CATCHMENTS = ['WD', 'SD', 'DD', 'TG', 'WG', 'FNQ', 'CQ', 'SEQ']
SOUTHERN_QLD = ['WD', 'SD', 'DD', 'TG', 'WG']

tr = transitions[transitions['transmission_catchment'].isin(QLD_CATCHMENTS)].copy()

# Transition counts: rows = release windows, cols = catchments
window_counts = (tr.groupby(['to_release','transmission_catchment']).size()
                 .unstack(fill_value=0)
                 .reindex(columns=QLD_CATCHMENTS, fill_value=0))

# Exposure: projects per catchment per release (= candidate population for transition)
exposure = (panel_clean.groupby(['release_date','transmission_catchment']).size()
            .unstack(fill_value=0)
            .reindex(columns=QLD_CATCHMENTS, fill_value=0))

print('Transition counts by (release window, catchment):')
print(window_counts)
print('\nProject exposure (count) by (release, catchment):')
print(exposure)

## Claim 1 — Over-dispersion of transition counts (Fano factor test)

**Null**: per-catchment transitions in each release window are Poisson with rate λ proportional to exposure. Fano = Var/Mean = 1.

**Alt**: counts are over-dispersed (Fano > 1) — some windows have unusually high transition activity.

### Fano on raw counts

First pass: Fano factor on the time series of transition counts for each catchment.
Caveat: this does not control for exposure (a catchment with growing project count will have Fano > 1 just from the growth trend).

In [ ]:
def fano_with_ci(counts, n_boot=2000, seed=0):
    """Fano factor with bootstrap 95% CI."""
    counts = np.asarray(counts)
    if counts.mean() == 0 or len(counts) < 3:
        return np.nan, (np.nan, np.nan)
    point = counts.var(ddof=1) / counts.mean()
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        s = rng.choice(counts, size=len(counts), replace=True)
        if s.mean() > 0:
            boots.append(s.var(ddof=1) / s.mean())
    if not boots:
        return point, (np.nan, np.nan)
    return point, (np.percentile(boots, 2.5), np.percentile(boots, 97.5))

fano_rows = []
for c in QLD_CATCHMENTS:
    counts = window_counts[c].values
    f, (lo, hi) = fano_with_ci(counts)
    fano_rows.append({
        'catchment': c, 'n_windows': len(counts),
        'total_transitions': int(counts.sum()), 'mean': counts.mean().round(2),
        'var': counts.var(ddof=1).round(2), 'fano': round(f, 2) if not np.isnan(f) else None,
        'ci_lo': round(lo, 2) if not np.isnan(lo) else None,
        'ci_hi': round(hi, 2) if not np.isnan(hi) else None,
    })
fano_df = pd.DataFrame(fano_rows)
print('Fano factor by catchment (raw counts):')
print(fano_df.to_string(index=False))
print('\nInterpretation:')
print('  Fano = 1 (CI includes 1): consistent with Poisson — no detectable clustering')
print('  Fano > 1 (CI above 1):    over-dispersed — clustering detected')
print('  Fano < 1 (CI below 1):    under-dispersed — regularised (e.g. queueing)')

### Exposure-controlled Fano

Raw Fano conflates herding with growth: if exposure doubles over the period, counts will trend up regardless of any clustering mechanism. Compute the *exposure-rate* (transitions per project at risk) per window, then check whether residuals from the catchment-specific mean rate are over-dispersed.

*This is the more conservative test.* It asks: holding the average transition rate constant, are some release windows in some catchments unusually high?

In [ ]:
# Per (release, catchment): observed transitions / projects-at-risk-in-prior-release
# 'projects at risk' for window ending at release R is exposure at the *previous* release
expo_lag = exposure.shift(1)  # exposure at start of each window
rate = (window_counts / expo_lag).replace([np.inf, -np.inf], np.nan)
rate = rate.dropna(how='all')  # drop the first window (no prior exposure)

print('Transition rate per project at risk:')
print(rate.round(3))

# Mean rate per catchment over the panel
mean_rate = rate.mean(axis=0)
print('\nMean rate per catchment (transitions per project per window):')
print(mean_rate.round(3).to_string())

# Expected counts: mean_rate * exposure
expected = expo_lag.dropna(how='all') * mean_rate
expected = expected.reindex_like(window_counts.loc[rate.index])
obs = window_counts.loc[rate.index]

# Per-catchment Fano on the obs-vs-expected residuals (normalised to Poisson scale)
# Under the rate model, var(obs - expected) / mean(expected) = 1 under Poisson
controlled_rows = []
for c in QLD_CATCHMENTS:
    o, e = obs[c].values, expected[c].values
    if e.mean() == 0:
        controlled_rows.append({'catchment': c, 'n_windows': len(o),
                                'total_obs': int(o.sum()), 'total_expected': round(float(e.sum()), 1),
                                'controlled_fano': None})
        continue
    # Variance of standardised residual (Pearson chi-square component)
    resid = (o - e) / np.sqrt(np.maximum(e, 1e-9))
    # Test whether resid is bigger than would be expected from Poisson sampling
    controlled_rows.append({
        'catchment': c, 'n_windows': len(o),
        'total_obs': int(o.sum()), 'total_expected': round(float(e.sum()), 1),
        'sum_sq_resid': round(float((resid**2).sum()), 2),
        'df': len(o) - 1,
        # Chi-sq with df=n-1: if Poisson, sum_sq_resid ~ chi2(df)
        'chi2_p': round(float(1 - stats.chi2.cdf((resid**2).sum(), df=len(o)-1)), 4),
    })
ctrl = pd.DataFrame(controlled_rows)
print('\nExposure-controlled test (Pearson chi-square of residuals):')
print(ctrl.to_string(index=False))
print('\nInterpretation:')
print('  chi2_p < 0.05: counts are over-dispersed beyond Poisson — clustering detected')
print('  chi2_p > 0.05: counts consistent with Poisson at given exposure')

In [ ]:
# Plot: Fano factor + 95% CI per catchment
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(fano_df))
fano_df_plot = fano_df.dropna(subset=['fano'])
x = np.arange(len(fano_df_plot))
ax.errorbar(
    x, fano_df_plot['fano'],
    yerr=[fano_df_plot['fano'] - fano_df_plot['ci_lo'],
          fano_df_plot['ci_hi'] - fano_df_plot['fano']],
    fmt='o', capsize=5, color='steelblue',
)
ax.axhline(1.0, color='grey', linestyle='--', linewidth=1, label='Poisson (Fano = 1)')
ax.set_xticks(x)
ax.set_xticklabels(fano_df_plot['catchment'])
ax.set_ylabel('Fano factor (variance / mean)')
ax.set_xlabel('Catchment')
ax.set_title('Claim 1 — Are transition counts over-dispersed?', loc='left', fontsize=12)
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

### Claim 1 interpretation

**Caveats first:** with 6 release windows per catchment, the bootstrap CI is wide. At n≈18 the bootstrap CIs are tighter and several catchments resolve from 'directionally consistent' to 'statistically detectable'. So absence of evidence is not evidence of absence — this notebook now uses the Wayback-recovered releases.

**What to look for in the result above:**
- Catchments where the **point estimate Fano is clearly > 1** are candidates for herding.
- Catchments where the **lower CI bound is > 1** are *statistically detectable* herding at this sample.
- The **exposure-controlled chi-square p-value** is the more disciplined test — it accounts for the fact that bigger catchments will have higher variance in raw counts.

Run the cells above to see what your specific data shows.

## Claim 3 — Spatial co-clustering (catchment concentration test)

**Null**: in each release window, the distribution of transitions across catchments matches the exposure distribution. I.e., a catchment with 20% of projects gets 20% of transitions.

**Alt**: transitions concentrate in *fewer* catchments than exposure-weighted random would predict.

**Test**: For each release window, run a chi-square test of observed counts against exposure-weighted expected counts. Combine the per-window p-values using Fisher's method to get an overall test.

In [ ]:
# Per release window: chi-square test of observed vs exposure-weighted expected
results = []
for window in rate.index:
    obs_row = window_counts.loc[window].values.astype(float)
    expo_row = expo_lag.loc[window].values.astype(float)
    n = obs_row.sum()
    if n < 3 or expo_row.sum() == 0:
        results.append({'window': window, 'total_transitions': int(n),
                        'chi2': None, 'p_value': None, 'skipped': 'n<3 or no exposure'})
        continue
    # Expected = n * (exposure share)
    expected_row = n * (expo_row / expo_row.sum())
    # Drop catchments where expected = 0
    mask = expected_row > 0
    if mask.sum() < 2:
        results.append({'window': window, 'total_transitions': int(n),
                        'chi2': None, 'p_value': None, 'skipped': 'too few non-zero cells'})
        continue
    chi2 = (((obs_row[mask] - expected_row[mask])**2) / expected_row[mask]).sum()
    df = mask.sum() - 1
    p = 1 - stats.chi2.cdf(chi2, df=df)
    # Top catchment(s) by observed/expected ratio
    rr = pd.Series(obs_row / np.maximum(expected_row, 1e-9), index=QLD_CATCHMENTS)
    top = rr[mask.tolist()].sort_values(ascending=False).head(3)
    results.append({
        'window': window, 'total_transitions': int(n),
        'chi2': round(float(chi2), 2), 'df': int(df),
        'p_value': round(float(p), 4),
        'concentrated_in': ', '.join(f'{c}({v:.1f}x)' for c, v in top.items()),
    })
cl3 = pd.DataFrame(results)
print('Per-window chi-square test (observed vs exposure-weighted expected):')
print(cl3.to_string(index=False))

# Fisher's method to combine
p_vals = cl3['p_value'].dropna().values
if len(p_vals) >= 2:
    # Make sure no p=0 (replace with smallest float)
    p_vals = np.maximum(p_vals, 1e-12)
    combined_stat = -2 * np.sum(np.log(p_vals))
    combined_p = 1 - stats.chi2.cdf(combined_stat, df=2 * len(p_vals))
    print(f"\nFisher's combined test (over {len(p_vals)} windows):")
    print(f'  Combined chi2 = {combined_stat:.2f}, df = {2*len(p_vals)}, p = {combined_p:.4f}')
    print(f"  Interpretation: p < 0.05 means transitions concentrate non-randomly across catchments.")

In [ ]:
# Plot: observed vs expected, per window
windows_to_plot = [w for w in rate.index if window_counts.loc[w].sum() >= 3]
n_plots = len(windows_to_plot)
if n_plots:
    fig, axes = plt.subplots(n_plots, 1, figsize=(10, 2.5 * n_plots), sharex=True)
    if n_plots == 1:
        axes = [axes]
    for ax, w in zip(axes, windows_to_plot):
        obs_row = window_counts.loc[w]
        expo_row = expo_lag.loc[w]
        n = obs_row.sum()
        exp_row = n * (expo_row / expo_row.sum())
        x = np.arange(len(QLD_CATCHMENTS))
        width = 0.35
        ax.bar(x - width/2, obs_row.values, width, label='Observed', color='steelblue')
        ax.bar(x + width/2, exp_row.values, width, label='Expected (exposure-weighted)', color='lightgrey')
        ax.set_xticks(x)
        ax.set_xticklabels(QLD_CATCHMENTS)
        ax.set_title(f'Window ending {w.date()} (n={int(n)} transitions)', loc='left', fontsize=10)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    axes[-1].set_xlabel('Catchment')
    fig.suptitle('Claim 3 — Observed vs exposure-weighted expected transitions', y=1.0)
    fig.tight_layout()
    plt.show()

### Claim 3 interpretation

**Caveats first:** chi-square tests need sufficient expected counts per cell (the standard rule is ≥5). With 1-10 transitions per window across 8 catchments, expected cells are often < 5 — the chi-square approximation may not be reliable. Exact tests (Fisher's exact, multinomial) would be more rigorous; we'd use them in a v1.0 of this analysis.

**What to look for:**
- Per-window p-values < 0.05: that window's transitions concentrated unexpectedly in one or two catchments.
- Fisher's combined p < 0.05: there's evidence of non-random concentration across the panel as a whole.
- `concentrated_in` column: identifies *which* catchments over-indexed in each window. Patterns here (e.g. WD over-indexing in 2021-07 and 2024-07) are the visible signature of synchronised buildout.

Run the cells above to see your data.

## Synthesis — what these tests tell you

After running, ask:

1. **Is the point-estimate Fano > 1 in your priority catchments (WD especially)?** Even if CIs are wide, a consistent pattern across multiple catchments is informative.
2. **Is the exposure-controlled chi-square significant for any catchment?** This is the more disciplined version of Claim 1.
3. **Does Fisher's combined test reject the null on Claim 3?** If so, transitions concentrate in catchments more than chance would predict — supporting the catchment-as-coupling-unit hypothesis.
4. **Which catchments over-index in which windows?** Look at the `concentrated_in` column for narrative material — when did WD pull more than its share of transitions?

**The honest summary** for v0.5 of the framework: with 7 releases, the tests have low statistical power. They can detect strong effects but not subtle ones. Getting more releases via Wayback Machine — and adding the constraint-binding overlay (Claim 4) — is what would lift this from suggestive to defensible.

## Next steps

1. *(Done — 22 releases including Wayback-recovered files form this analysis.)*
2. Test Claim 2 (Layer A events → transition rate increase) — needs verified Layer A event dates first.
3. Test Claim 4 (synchronisation → constraint binding) — needs constraint-binding events tagged to catchments via your `nem-constraints` work.
4. If Claims 1-4 hold up, test Claim 5 (out-of-sample prediction) — needs a holdout split.